In [ ]:
# ======================================================
# NOTEBOOK: PATHWAY INFERENCE PIPELINE (TRACK A COMPLIANT)
# ======================================================

In [ ]:
# !pip install -q \
#   "numpy>=1.24,<2.1.0" \
#   "pandas<2.2.0" \
#   pathway \
#   transformers \
#   accelerate \
#   sentence-transformers \
#   tqdm

In [ ]:
!pip -q install pathway

In [21]:
# --- CELL 1: SETUP ---

import pickle
import pandas as pd
import torch
import torch.nn.functional as F
import pathway as pw
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

print(f"Pathway version: {pw.__version__}")
print(f"Torch version: {torch.__version__}")
print("✅ All imports successful!")

# --- IMPORT METRICS (Add this early in your notebook) ---
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    f1_score, 
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ All metrics libraries imported successfully")

Pathway version: 0.27.1
Torch version: 2.8.0+cu126
✅ All imports successful!
✓ All metrics libraries imported successfully


In [23]:
# --- CELL 2: CONFIGURATION ---
# UPDATE THESE PATHS TO WHERE YOUR .PKL FILES ARE LOCATED
CASTAWAYS_PKL = "/kaggle/input/kdsh26-in-search-of-the-castaways-processeddataset/castaways_chunks.pkl"
MONTECRISTO_PKL = "/kaggle/input/kdsh26-the-count-of-monte-cristo-processeddataset/montecristo_chunks.pkl"
CASTAWAYS_EMB = "/kaggle/input/kdsh26-in-search-of-the-castaways-processeddataset/castaways_embeddings_qwen.pkl"
MONTECRISTO_EMB = "/kaggle/input/kdsh26-the-count-of-monte-cristo-processeddataset/montecristo_embeddings_qwen.pkl"
TEST_CSV = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv"
SUBMISSION_FILE = "submission.csv"
VALIDATION_FILE = "validation_results.csv"

# Training file for Validation
TRAIN_CSV = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/train.csv"

# Models
EMBED_MODEL_ID = "Alibaba-NLP/gte-Qwen2-7B-instruct"
LLM_MODEL_ID = "Qwen/Qwen2.5-72B-Instruct-AWQ" 
# Use "Qwen/Qwen2.5-14B-Instruct" if 72B OOMs or fails on T4

In [3]:
# --- CELL 3: MODEL HELPERS ---
def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

print("Loading Embedding Model...")
try:
    emb_config = AutoConfig.from_pretrained(EMBED_MODEL_ID, trust_remote_code=True)
    emb_config.use_cache = False # Critical Fix
    embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID, trust_remote_code=True)
    embed_model = AutoModel.from_pretrained(EMBED_MODEL_ID, config=emb_config, trust_remote_code=True, torch_dtype=torch.float16).to("cuda")
    embed_model.eval()
except Exception as e:
    print(f"Embedding Model Load Failed: {e}")


Loading Embedding Model...


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model-00001-of-00007.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00006-of-00007.safetensors:   0%|          | 0.00/3.66G [00:00<?, ?B/s]

model-00004-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00005-of-00007.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
# --- CELL 4: LOAD DATA (HYBRID APPROACH) ---
print("Loading Processed Data...")

# 1. Load Texts via Pandas (Standard)
df_cast = pd.read_pickle(CASTAWAYS_PKL)
df_monte = pd.read_pickle(MONTECRISTO_PKL)

# 2. Load Vectors via Pickle
with open(CASTAWAYS_EMB, "rb") as f:
    vec_cast = pickle.load(f)
with open(MONTECRISTO_EMB, "rb") as f:
    vec_monte = pickle.load(f)

# 3. Create Fast Lookup
BOOK_DATA = {
    "In Search of the Castaways": (df_cast, torch.tensor(vec_cast).cuda()),
    "The Count of Monte Cristo": (df_monte, torch.tensor(vec_monte).cuda())
}

Loading Processed Data...


/tmp/ipykernel_445/1106753722.py:16: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  "In Search of the Castaways": (df_cast, torch.tensor(vec_cast).cuda()),


In [5]:
# --- CELL 5: PATHWAY INTEGRATION (INGESTION) ---
# We fulfill Track A requirement by ingesting queries via Pathway
print("Ingesting Queries via Pathway...")
test_df = pd.read_csv(TEST_CSV)
query_table = pw.debug.table_from_pandas(test_df[['id', 'content', 'book_name']])

# Trigger ingestion (Static snapshot for notebook)
query_df_processed = pw.debug.table_to_pandas(query_table)
print(f"Pathway ingested {len(query_df_processed)} queries.")

Ingesting Queries via Pathway...
Pathway ingested 60 queries.


In [6]:
# --- CELL 6: EMBED QUERIES (PYTHON BATCH) ---
def embed_queries_batch(queries):
    # Add Instruction for Query Side
    instruct_queries = [
        f"Instruct: Given a user query, retrieve relevant passages from the novel.\nQuery: {q}" 
        for q in queries
    ]
    all_vecs = []
    batch_size = 4
    for i in tqdm(range(0, len(instruct_queries), batch_size), desc="Embedding Queries"):
        batch = instruct_queries[i:i+batch_size]
        inputs = embed_tokenizer(batch, max_length=512, padding=True, truncation=True, return_tensors='pt').to("cuda")
        with torch.no_grad():
            out = embed_model(**inputs, use_cache=False)
            vecs = last_token_pool(out.last_hidden_state, inputs['attention_mask'])
            vecs = F.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs)
    return torch.cat(all_vecs)

print("Embedding queries...")
query_vectors = embed_queries_batch(query_df_processed['content'].tolist())

Embedding queries...


Embedding Queries: 100%|██████████| 15/15 [00:01<00:00, 11.98it/s]


In [ ]:
!pip -q install autoawq

In [ ]:
!pip -q install bitsandbytes

In [9]:
# --- CELL 7: SEARCH & JUDGE (FIXED: USING QWEN 14B) ---
from transformers import BitsAndBytesConfig

# SWITCH TO 14B (Fits on Disk & GPU easily)
LLM_MODEL_ID_BASE = "Qwen/Qwen2.5-14B-Instruct" 

print(f"Loading Judge LLM: {LLM_MODEL_ID_BASE} with BitsAndBytes 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID_BASE, trust_remote_code=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID_BASE,
    quantization_config=bnb_config, 
    device_map="auto", 
    trust_remote_code=True
)

final_predictions = []
final_rationales = []
ids = []

print("Running Inference...")

# FIX: Use 'enumerate' to get 'i' (0, 1, 2...) for the Tensor
for i, (_, row) in tqdm(enumerate(query_df_processed.iterrows()), total=len(query_df_processed)):
    
    # Use 'i' (integer), not 'idx' (pathway pointer)
    q_vec = query_vectors[i].unsqueeze(0) # [1, Dim]
    
    target_book = row['book_name']
    claim = row['content']
    q_id = row['id']
    
    # Validation
    if target_book not in BOOK_DATA:
        ids.append(q_id)
        final_predictions.append("consistent") # Safe fallback
        final_rationales.append("Book data not found")
        continue
        
    df_book, emb_book = BOOK_DATA[target_book]
    
    # Similarity Search (Dot Product on GPU)
    scores = torch.mm(q_vec, emb_book.T).squeeze(0)
    top_k = torch.topk(scores, k=5)
    
    # Retrieve Context
    chunks = []
    for doc_idx in top_k.indices.cpu().numpy():
        chunks.append(df_book.iloc[doc_idx]['text'])
        
    context_str = "\n---\n".join(chunks)
    
    # Reasoning Prompt
    prompt = f"""<|im_start|>system
You are a logic judge. Check if the Claim contradicts the Story Context.
Output ONLY 'consistent' or 'contradict'.<|im_end|>
<|im_start|>user
Context:
{context_str}

Claim:
{claim}

Verdict:<|im_end|>
<|im_start|>assistant
"""
    inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=10, do_sample=False)
    
    ans = llm_tokenizer.decode(out[0], skip_special_tokens=True).split("Verdict:")[-1].strip().lower()
    
    pred = "contradict" if "contradict" in ans else "consistent"
    
    ids.append(q_id)
    final_predictions.append(pred)
    final_rationales.append(f"Checked top {len(chunks)} relevant excerpts.")



Loading Judge LLM: Qwen/Qwen2.5-14B-Instruct with BitsAndBytes 4-bit...


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Running Inference...


100%|██████████| 60/60 [00:26<00:00,  2.26it/s]


In [10]:
# --- CELL 8: SAVE ---
sub = pd.DataFrame({'id': ids, 'prediction': final_predictions, 'rationale': final_rationales})
sub.to_csv(SUBMISSION_FILE, index=False)
print(f"DONE! Submission saved to {SUBMISSION_FILE}")

DONE! Submission saved to submission.csv


In [13]:
# ____ VALIDATION ____

In [19]:
def run_inference(query_df, query_vectors, is_validation=False):
    """
    Run inference on a DataFrame of queries.
    
    Args:
        query_df: DataFrame with 'id', 'content', 'book_name' columns
        query_vectors: Pre-computed query embeddings tensor
        is_validation: If True, expect 'label' column for accuracy calculation
    
    Returns:
        results_df: DataFrame with predictions and optionally metrics
    """
    predictions = []
    rationales = []
    ids = []
    scores_list = []  # Track retrieval scores
    
    print(f"Running Inference on {len(query_df)} queries...")
    
    for i, (_, row) in tqdm(enumerate(query_df.iterrows()), total=len(query_df)):
        q_vec = query_vectors[i].unsqueeze(0)  # [1, Dim]
        target_book = row['book_name']
        claim = row['content']
        q_id = row['id']
        
        # Validation: Check if book data exists
        if target_book not in BOOK_DATA:
            ids.append(q_id)
            predictions.append("consistent")
            rationales.append("Book data not found")
            scores_list.append(0.0)
            continue
            
        df_book, emb_book = BOOK_DATA[target_book]
        
        # Similarity Search (Cosine Similarity on GPU)
        scores = torch.mm(q_vec, emb_book.T).squeeze(0)
        top_k = torch.topk(scores, k=5)
        
        # Retrieve Top-K Context Chunks
        chunks = []
        top_score = top_k.values[0].item() if len(top_k.values) > 0 else 0.0
        
        for doc_idx in top_k.indices.cpu().numpy():
            chunks.append(df_book.iloc[doc_idx]['text'])
            
        context_str = "\n---\n".join(chunks)
        
        # LLM Judge: Reason about contradiction
        prompt = f"""<|im_start|>system
You are a logic judge. Your task is to determine if the Claim contradicts the Story Context.
- If the Claim contradicts facts/events in the Context, output: 'contradict'
- If the Claim is consistent with the Context, output: 'consistent'
Output ONLY one word: 'consistent' or 'contradict'<|im_end|>
<|im_start|>user
Story Context:
{context_str}

Claim:
{claim}

Verdict:<|im_end|>
<|im_start|>assistant
"""
        # Generate prediction
        try:
            inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")
            with torch.no_grad():
                out = llm_model.generate(**inputs, max_new_tokens=10, do_sample=False)
            
            # FIXED: Decode only the generated tokens, not the full output
            ans = llm_tokenizer.decode(out[0], skip_special_tokens=True).split("Verdict:")[-1].strip().lower()
            pred = "contradict" if "contradict" in ans else "consistent"
        except Exception as e:
            print(f"Error processing query {q_id}: {e}")
            pred = "consistent"  # Safe default
            ans = f"error: {str(e)}"
        
        # Append results
        ids.append(q_id)
        predictions.append(pred)
        rationales.append(
            f"Checked top {len(chunks)} relevant excerpts. "
            f"Max retrieval score: {top_score:.4f}. "
            f"LLM verdict: {ans}"
        )
        scores_list.append(top_score)
    
    # Build results DataFrame (FIXED: indentation)
    results_df = pd.DataFrame({
        'id': ids,
        'prediction': predictions,
        'rationale': rationales,
        'retrieval_score': scores_list
    })
    
    return results_df


In [24]:
# --- CELL 10: VALIDATION ON TRAIN SET ---
print("=== RUNNING VALIDATION ON TRAIN SET ===")

# Load and ingest training data via Pathway
train_df = pd.read_csv(TRAIN_CSV)
print(f"Train set: {len(train_df)} queries")

train_table = pw.debug.table_from_pandas(train_df[['id', 'content', 'book_name', 'label']])
train_df_pathway = pw.debug.table_to_pandas(train_table)

print(f"Pathway ingested {len(train_df_pathway)} training queries")

# Embed training queries
print("...Embedding training queries...")
train_vectors = embed_queries_batch(train_df_pathway['content'].tolist())

# Run inference on training set
train_results = run_inference(train_df_pathway, train_vectors, is_validation=True)

# Add ground truth labels
train_results['label'] = train_df_pathway['label'].values

# Calculate metrics
accuracy = accuracy_score(train_results['label'], train_results['prediction'])
print(f"=== VALIDATION METRICS ===")
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(train_results['label'], train_results['prediction']))

# Confusion Matrix
cm = confusion_matrix(train_results['label'], train_results['prediction'])
print("Confusion Matrix:")
print(cm)

# Save validation results
train_results.to_csv(VALIDATION_FILE, index=False)
print(f"Validation results saved to {VALIDATION_FILE}")


=== RUNNING VALIDATION ON TRAIN SET ===
Train set: 80 queries
Pathway ingested 80 training queries
...Embedding training queries...


Embedding Queries: 100%|██████████| 20/20 [00:00<00:00, 53.99it/s]


Running Inference on 80 queries...


100%|██████████| 80/80 [00:33<00:00,  2.40it/s]

=== VALIDATION METRICS ===
Accuracy: 0.3875
Classification Report:
              precision    recall  f1-score   support

  consistent       0.67      0.08      0.14        51
  contradict       0.36      0.93      0.52        29

    accuracy                           0.39        80
   macro avg       0.52      0.50      0.33        80
weighted avg       0.56      0.39      0.28        80

Confusion Matrix:
[[ 4 47]
 [ 2 27]]
Validation results saved to validation_results.csv


In [26]:
# --- CELL 11: ANALYSIS ---
print("=== DETAILED ANALYSIS ===")
print(train_results.groupby('label')['prediction'].value_counts().unstack(fill_value=0))

# Find misclassified examples
misclassified = train_results[train_results['label'] != train_results['prediction']]
print(f"Misclassified: {len(misclassified)}/{len(train_results)}")
if len(misclassified) > 0:
    print("First 5 Misclassified Examples:")
    print(misclassified.head(5)[['id', 'label', 'prediction', 'rationale']])


=== DETAILED ANALYSIS ===
prediction  consistent  contradict
label                             
consistent           4          47
contradict           2          27
Misclassified: 49/80
First 5 Misclassified Examples:
    id       label  prediction  \
0   67  consistent  contradict   
1  125  consistent  contradict   
2  126  consistent  contradict   
5   65  consistent  contradict   
6   50  consistent  contradict   

                                           rationale  
0  Checked top 5 relevant excerpts. Max retrieval...  
1  Checked top 5 relevant excerpts. Max retrieval...  
2  Checked top 5 relevant excerpts. Max retrieval...  
5  Checked top 5 relevant excerpts. Max retrieval...  
6  Checked top 5 relevant excerpts. Max retrieval...  
